In [26]:
import numpy as np
from pandocfilters import attributes
from scipy.cluster.hierarchy import linkage

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import infoclus

In [46]:
import os
from config import PROJECT_ROOT
from src.caching import from_cache

emb_name = 'tsne'
linkage = 'single'
modify_hierarchical = True
data_name = 'cytometry_2500'
file_path = os.path.join(PROJECT_ROOT, 'data', data_name, 'cache', emb_name +'_'+ linkage + '_' + 'modify_' + str(modify_hierarchical))

if os.path.exists(file_path):
    print('loading ' + file_path)
    infoclus_obj = from_cache(file_path)
    print('done')
else:
    infoclus_obj = infoclus.InfoClus(dataset_name=data_name, emb_name=emb_name, linkage=linkage, modify_hierarchical=modify_hierarchical)

loading C:\Users\fulai\PycharmProjects\InfoClus\data\cytometry_2500\cache\tsne_single_modify_True
done


In [45]:
infoclus_obj.optimise()


splitting by nodes start ... 
37 iterations done.

InfoClus - Dataset: cytometry_2500 Emb: tsne Alpha: 250 Beta: 1.5 Ref. Runtime: 30
checking if the sum of clusters idxes equals to the data size: Count of Clusters: 4
    cluster 0:
        count of points: 294
        attributes: CD11c CD11b 
    cluster 1:
        count of points: 1534
        attributes: CD19 CD3 MHCII 
    cluster 2:
        count of points: 618
        attributes: CD3 CD19 MHCII 
    cluster 3:
        count of points: 54
        attributes: CD64 Autofluo. 
SI:  23.36335313735609


In [20]:
from sklearn.discriminant_analysis import StandardScaler
import infoclus_utils
import numpy as np

data = infoclus_obj.data_obj.data_raw.values
scaler = StandardScaler()
data = scaler.fit_transform(data)
prior = [np.mean(data, axis=0), np.var(data, axis=0)]
# prior = infoclus_obj.model_obj.prior
clustering = infoclus_obj.result_obj.clustering
attributes = infoclus_obj.result_obj.attributes_opt
count_clusters = infoclus_obj.result_obj.count_clusters
ic = 0

for i in range(count_clusters):
    cluster = data[np.where(clustering == i)]
    mean_cluster = np.mean(cluster, axis=0)
    var_cluster = np.var(cluster, axis=0)
    kl_cluster = infoclus_utils.kl_gaussian(mean_cluster, var_cluster, prior[0], prior[1])
    for attr in attributes[i]:
        ic = ic + len(cluster) * kl_cluster[attr]

c = infoclus_obj.alpha + (2*sum(len(att) for att in attributes))**infoclus_obj.beta
si = ic/c

si

np.float64(9.714947062152278)

In [59]:
a = infoclus_obj.data_obj.data.columns[8]

In [22]:
np.mean(data, axis=0)

array([-1.55215646e-16,  5.51877853e-16, -3.79416024e-16, -1.72461829e-16,
       -1.37969463e-16,  6.89847316e-16,  1.72461829e-17, -6.12239493e-16,
       -1.72461829e-15,  8.19193688e-16,  2.01780340e-15, -1.72461829e-16,
        1.72461829e-17, -1.89708012e-16, -3.10431292e-16, -3.44923658e-16,
        8.62309145e-18, -4.31154573e-17, -5.94993310e-16, -5.43254762e-16,
       -6.89847316e-17, -3.44923658e-16,  3.01808201e-16, -3.44923658e-17,
        1.20723280e-16, -7.41585865e-16,  4.13908390e-16, -3.62169841e-16,
        1.20723280e-16,  2.58692744e-17,  6.03616402e-17])

In [24]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42).fit(data)